# Add SageMaker training and S3 integration



## Startup cells

In [0]:
# Set environment variables for sagemaker_studio imports

import os
os.environ['DataZoneProjectId'] = '5y1nkdhiat07fr'
os.environ['DataZoneDomainId'] = 'dzd-baa51un8enpy07'
os.environ['DataZoneEnvironmentId'] = '6g07zm1q2jqsyv'
os.environ['DataZoneDomainRegion'] = 'us-east-1'

# create both a function and variable for metadata access
_resource_metadata = None

def _get_resource_metadata():
    global _resource_metadata
    if _resource_metadata is None:
        _resource_metadata = {
            "AdditionalMetadata": {
                "DataZoneProjectId": "5y1nkdhiat07fr",
                "DataZoneDomainId": "dzd-baa51un8enpy07",
                "DataZoneEnvironmentId": "6g07zm1q2jqsyv",
                "DataZoneDomainRegion": "us-east-1",
            }
        }
    return _resource_metadata
metadata = _get_resource_metadata()

In [0]:
"""
Logging Configuration

Purpose:
--------
This sets up the logging framework for code executed in the user namespace.
"""

from typing import Optional


def _set_logging(log_dir: str, log_file: str, log_name: Optional[str] = None):
    import os
    import logging
    from logging.handlers import RotatingFileHandler

    level = logging.INFO
    max_bytes = 5 * 1024 * 1024
    backup_count = 5

    # fallback to /tmp dir on access, helpful for local dev setup
    try:
        os.makedirs(log_dir, exist_ok=True)
    except Exception:
        log_dir = "/tmp/kernels/"

    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, log_file)

    logger = logging.getLogger() if not log_name else logging.getLogger(log_name)
    logger.handlers = []
    logger.setLevel(level)

    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")

    # Rotating file handler
    fh = RotatingFileHandler(filename=log_path, maxBytes=max_bytes, backupCount=backup_count, encoding="utf-8")
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    logger.info(f"Logging initialized for {log_name}.")


_set_logging("/var/log/computeEnvironments/kernel/", "kernel.log")
_set_logging("/var/log/studio/data-notebook-kernel-server/", "metrics.log", "metrics")

In [0]:
import logging
from sagemaker_studio import ClientConfig, sqlutils, sparkutils, dataframeutils

logger = logging.getLogger(__name__)
logger.info("Initializing sparkutils")
spark = sparkutils.init()
logger.info("Finished initializing sparkutils")

In [0]:
def _reset_os_path():
    """
    Reset the process's working directory to handle mount timing issues.
    
    This function resolves a race condition where the Python process starts
    before the filesystem mount is complete, causing the process to reference
    old mount paths and inodes. By explicitly changing to the mounted directory
    (/home/sagemaker-user), we ensure the process uses the correct, up-to-date
    mount point.
    
    The function logs stat information (device ID and inode) before and after
    the directory change to verify that the working directory is properly
    updated to reference the new mount.
    
    Note:
        This is executed at module import time to ensure the fix is applied
        as early as possible in the kernel initialization process.
    """
    try:
        import os
        import logging

        logger = logging.getLogger(__name__)
        logger.info("---------Before------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)

        os.chdir("/home/sagemaker-user")

        logger.info("---------After------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)
    except Exception as e:
        logger.exception(f"Failed to reset working directory: {e}")

_reset_os_path()

## Notebook

In [0]:
import boto3
s3 = boto3.client("s3")
response = s3.list_buckets()

[b["Name"] for b in response["Buckets"]]

['amazon-sagemaker-153058521958-us-east-1-5y1nkdhiat07fr',
 'cloud-ml-genai-api-proj-153058521958-us-east-1-an']

In [0]:
import pandas as pd

bucket = "cloud-ml-genai-api-proj-153058521958-us-east-1-an"

df = pd.read_csv( f"s3://{bucket}/data/raw/creditcard.csv")

df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,-0.551600,-0.617801,-0.991390,-0.311169,1.468177,-0.470401,0.207971,0.025791,0.403993,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,1.612727,1.065235,0.489095,-0.143772,0.635558,0.463917,-0.114805,-0.183361,-0.145783,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,0.624501,0.066084,0.717293,-0.165946,2.345865,-2.890083,1.109969,-0.121359,-2.261857,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,-0.226487,0.178228,0.507757,-0.287924,-0.631418,-1.059647,-0.684093,1.965775,-1.232622,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,-0.822843,0.538196,1.345852,-1.119670,0.175121,-0.451449,-0.237033,-0.038195,0.803487,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import mlflow
import mlflow.sklearn

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)

mlflow.set_experiment("fraud_detection_sagemaker")

with mlflow.start_run():

    model = RandomForestClassifier(n_estimators=100, random_state=42)

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    report = classification_report(y_test, preds, output_dict=True)

    mlflow.log_metric(
        "f1_score", report["1"]["f1-score"]
    )

    mlflow.sklearn.log_model( model, "fraud_model")

    print(report)

/sagemaker_packages/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026/05/11 15:25:52 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/05/11 15:25:52 INFO mlflow.store.db.utils: Updating database tables


2026/05/11 15:25:53 INFO mlflow.tracking.fluent: Experiment with name 'fraud_detection_sagemaker' does not exist. Creating a new experiment.


2026/05/11 15:29:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/05/11 15:29:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'0': {'precision': 0.9995956754856289, 'recall': 0.9999648283624085, 'f1-score': 0.999780217848069, 'support': 56864.0}, '1': {'precision': 0.974025974025974, 'recall': 0.7653061224489796, 'f1-score': 0.8571428571428571, 'support': 98.0}, 'accuracy': 0.9995611109160493, 'macro avg': {'precision': 0.9868108247558014, 'recall': 0.882635475405694, 'f1-score': 0.928461537495463, 'support': 56962.0}, 'weighted avg': {'precision': 0.9995516842152548, 'recall': 0.9995611109160493, 'f1-score': 0.999534818084207, 'support': 56962.0}}


In [0]:
import os
import mlflow

bucket = "cloud-ml-genai-api-proj-153058521958-us-east-1-an"

mlflow.set_tracking_uri("file:///tmp/mlruns")
mlflow.set_experiment("fraud_detection_sagemaker")

print("Tracking URI:", mlflow.get_tracking_uri())

/sagemaker_packages/.venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/11 15:37:39 INFO mlflow.tracking.fluent: Experiment with name 'fraud_detection_sagemaker' does not exist. Creating a new experiment.


Tracking URI: file:///tmp/mlruns


In [0]:
import joblib
import boto3

model_path = "/tmp/fraud_model.pkl"
joblib.dump(model, model_path)

s3 = boto3.client("s3")

s3.upload_file(model_path, bucket, "models/fraud_model.pkl")

print(f"Uploaded model to s3://{bucket}/models/fraud_model.pkl")

Uploaded model to s3://cloud-ml-genai-api-proj-153058521958-us-east-1-an/models/fraud_model.pkl


In [0]:
# Validate the saved model

response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix="models/"
)

for obj in response.get("Contents", []):
    print(obj["Key"], obj["Size"])

models/ 0
models/fraud_model.pkl 2615641


In [0]:
# syncing local Mlruns to s3 

!aws s3 sync ./mlruns s3://cloud-ml-genai-api-proj-153058521958-us-east-1-an/mlflow/

## Shutdown cells

In [0]:
"""
Stop spark session and associated Athena Spark session
"""

from IPython import get_ipython as _get_ipython
_get_ipython().user_ns["spark"].stop()